# ⚡ QLoRA 4-bit Fine-Tuning Studio (Colab T4 Free)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SudhirDevOps1/t4-gpu-llm-api/blob/main/colab/07_qlora_training_studio.ipynb)
[![Notebook Store](https://img.shields.io/badge/Notebook_Store-Obsidian_Hub-10b981?style=for-the-badge&logo=github)](https://github.com/SudhirDevOps1/t4-gpu-llm-api)

> **This notebook actually trains.** It loads `Qwen2.5-0.5B-Instruct` in 4-bit NF4, trains a LoRA adapter
> on 300 examples from `timdettmers/openassistant-guanaco`, runs a test generation, and downloads the
> adapter as a zip. (Earlier versions only printed a config object without training anything.)

- **Time:** ~5–10 minutes on a free T4 for 300 examples, 1 epoch
- **VRAM:** ~4 GB (0.5B model in NF4 + LoRA)
- **Output:** a `~/qwen-qlora/adapter` directory with `adapter_model.safetensors`
- Swap `BASE_MODEL` for a 7B/8B model if you have more VRAM headroom.


In [ ]:
#@title 🚀 1. Hardware Check & Session Fingerprint
import hashlib, os, socket, time, uuid

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

raw_fp = f"colab-qlora-{socket.gethostname()}-{uuid.uuid4()}-{time.time()}"
INSTANCE_ID = "colab-qlora-" + hashlib.sha256(raw_fp.encode()).hexdigest()[:8]
print(f"🆔 Unique QLoRA Instance Fingerprint: {INSTANCE_ID}")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv


In [ ]:
#@title 📦 2. Install Pinned Fine-Tuning Stack
# Pinned together because TRL's SFT API moved between 0.11 -> 0.12.
!pip install -q -U "bitsandbytes>=0.44" "peft==0.13.2" "trl==0.12.2" "transformers==4.46.3" "datasets==3.0.1" "accelerate==1.0.1"
print("✅ PEFT / bitsandbytes / TRL installed.")


In [ ]:
#@title 🧠 3. Load 4-bit Model + LoRA Config
import os

import torch
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
WORK_DIR = "/content/qwen-qlora" if os.path.isdir("/content") else "/tmp/qwen-qlora"
ADAPTER_DIR = os.path.join(WORK_DIR, "adapter")
os.makedirs(WORK_DIR, exist_ok=True)

if not torch.cuda.is_available():
    raise SystemExit("❌ No GPU. In Colab: Runtime -> Change runtime type -> T4 GPU.")

print("📥 Loading dataset slice (300 examples)...")
dataset = load_dataset("timdettmers/openassistant-guanaco", split="train[:300]")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
print(f"✅ {BASE_MODEL} loaded in 4-bit NF4. Adapter will be written to {ADAPTER_DIR}")


In [ ]:
#@title 🏋️ 4. Train the LoRA Adapter
import torch
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=WORK_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=5,
    save_strategy="no",
    max_seq_length=512,
    dataset_text_field="text",
    report_to="none",
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

trainer.train()
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✅ Adapter saved to {ADAPTER_DIR}")


In [ ]:
#@title 🧪 5. Test the Adapter & Download It
import glob
import shutil

import torch

model = trainer.model
model.config.use_cache = True
model.eval()

prompt = "### Human: Explain in one sentence what a GPU is used for.\n### Assistant:"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.7, top_p=0.9)
print(tokenizer.decode(output[0], skip_special_tokens=True))

archive = shutil.make_archive(WORK_DIR.rstrip("/") + "_adapter", "zip", ADAPTER_DIR)
print(f"📦 Adapter zip: {archive} ({os.path.getsize(archive) / 1024**2:.1f} MB)")

try:
    from google.colab import files
    files.download(archive)
except Exception:
    print("Not running on Colab — download the zip manually from:", archive)
